# Figure 1 — convergence to fast-excursion Heston

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from fast_excursion_limit import plot_config
from fast_excursion_limit.heston_random_ode import HestonRandomODE

plot_config.set_style()

SAVE_PLOT = True

## Parameters

In [ ]:
from fast_excursion_limit import defaults

reversions = [1, 10, 100, 1000]

seeds = [10, 20]

maturity = defaults.MATURITY_PATHWISE
step_size = defaults.DELTA_SCALE_PATHWISE * maturity

ode_model = HestonRandomODE.example()
feh_model = ode_model.fast_excursion_limit()
print(feh_model)

## Simulate paths

In [ ]:
feh_paths = {}
ode_paths = {}

for seed in seeds:
    np.random.seed(seed)
    Z_feh, Y_feh = feh_model.simulate(maturity=maturity, step_size=step_size)
    feh_paths[seed] = (Y_feh, Z_feh)
    for reversion in reversions:
        ode_model.reversion = reversion
        np.random.seed(seed)
        Z_ode, Y_ode = ode_model.simulate(maturity=maturity, step_size=step_size)
        ode_paths[(reversion, seed)] = (Y_ode, Z_ode)

## Plot

In [ ]:
from matplotlib.legend_handler import HandlerTuple

nrows, ncols = 2, 2
colors = plot_config.COLORS[:2]  # teal, purple

fig, axes = plt.subplots(nrows, ncols, figsize=(6.5, 6.5), sharex=True, sharey=True)
for ax, reversion in zip(axes.flat, reversions):
    ax.set_box_aspect(1)
    exponent = round(np.log10(reversion))

    feh_lines = []
    for i, seed in enumerate(seeds):
        Y_feh, Z_feh = feh_paths[seed]
        (line,) = ax.plot(Y_feh, Z_feh, color=colors[i], alpha=0.4)
        feh_lines.append(line)

    ode_lines = []
    for i, seed in enumerate(seeds):
        Y_ode, Z_ode = ode_paths[(reversion, seed)]
        (line,) = ax.plot(Y_ode, Z_ode, color=colors[i], alpha=1)
        ode_lines.append(line)

    ax.legend(
        [tuple(ode_lines), tuple(feh_lines)],
        [rf"Classical (reversion: $10^{exponent}$)", "Fast-excursion limit"],
        handler_map={tuple: HandlerTuple(ndivide=None)},
        loc="best",
    )

for ax in axes[-1, :]:
    ax.set_xlabel("Time")
for ax in axes[:, 0]:
    ax.set_ylabel("Price")

fig.tight_layout()
if SAVE_PLOT:
    fig.savefig(plot_config.PLOTS_DIR / "figure-1.pdf")